# Week 5 — Engineer Features and Reduce Dimensionality

**===== WEEK 5 START =====**

**Project:** Airline Passenger Demand Forecasting, BTS T-100 Domestic Segment

**Week 5 objectives**
- Generate cross-field domain features from current-month carrier-route data.
- Decide whether synthetic augmentation is appropriate.
- Reduce the numeric feature space by principal component analysis (PCA) **fitted on training only**. 

> A forecast for month *t+1* uses observed information from month *t*. The exercise assumes the month-*t* BTS aggregates are available at the time the forecast is issued. Operational reporting latency must be checked before deployment.

## 1. Environment / reusable code

- All project logic is in `Src/week5_src_features_dimensionality.py`.
- Notebook imports the same module so that it and the standalone Python program have identical formulas and preprocessing.

In [ ]:
from pathlib import Path
import importlib.util
import sys
from IPython.display import display, Image


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for folder in (start, *start.parents):
        if (folder / "Data").is_dir() and (folder / "Src").is_dir() and (folder / "Notebooks").is_dir():
            return folder
    raise FileNotFoundError("Could not locate the repository root.")

PROJECT_ROOT = find_project_root()
TABLES_DIR = PROJECT_ROOT / "Reports" / "week5" / "week5_tables"
FIGURES_DIR = PROJECT_ROOT / "Reports" / "week5" / "week5_figures"
PROCESSED_DIR = PROJECT_ROOT / "Data" / "processed" / "week5"
module_path = PROJECT_ROOT / "Src" / "week5_src_features_dimensionality.py"
spec = importlib.util.spec_from_file_location("week5_features", module_path)
week5 = importlib.util.module_from_spec(spec)
sys.modules["week5_features"] = week5
spec.loader.exec_module(week5)
print("Repository:", PROJECT_ROOT)
print("Source:", module_path)

## 2. Rebuild Week 2/3 data

- Use the original BTS T-100 ZIP archives and Week 2 ingestion code. 
- Week 3 target eligibility and chronological 70/15/15 split are reused. 
- The model target is the **following month's passengers for the same carrier and directional route**.

In [ ]:
monthly = week5.load_monthly_without_rewriting_reports(PROJECT_ROOT)
print(f"Carrier-route-month observations: {len(monthly):,}")
print("Reference months:", monthly['Period'].min(), "through", monthly['Period'].max())
assert all(x in monthly.columns for x in week5.CANDIDATE_FEATURES[:5])

## 3. Engineer four cross-field features

| New feature | Formula | Intended signal |
|---|---|---|
| `SeatsPerFlight` | `Seats / DepPerformed` | Mean offered seat capacity per performed departure |
| `PassengersPerFlight` | `Passengers / DepPerformed` | Mean passengers per performed departure |
| `UnusedSeats` | `Seats - Passengers` if nonnegative | Current-month unused reported seats |
| `PassengerMiles` | `Passengers * Distance` | Passenger-mile exposure combining demand and route length |

Each feature combines **two original input fields**. Zero performed departures produces a missing ratio; negative `UnusedSeats` is treated as invalid/missing rather than silently retained. The next-month target is **not** used in any feature calculation.

In [ ]:
features = week5.create_cross_field_features(monthly)
display(week5._feature_definitions())
display(features[["Period", *week5.NEW_FEATURES]].head(8))

## 4. Preserve the Week 3 train/validation/test partitions

In [ ]:
splits, split_summary = week5.get_week3_splits(features, PROJECT_ROOT)
display(split_summary)
assert splits["Training"]["TargetPeriod"].max() < splits["Validation"]["TargetPeriod"].min()
assert splits["Validation"]["TargetPeriod"].max() < splits["Test"]["TargetPeriod"].min()

## 5. PCA on nine candidate numeric predictors

The five original numeric features and four engineered features make **nine candidate inputs**. Categorical IDs, dates, and `TargetPassengersNextMonth` are excluded. We apply deterministic `log1p`, impute with **training** medians, remove training-constant columns, standardize with **training** parameters, then fit PCA **on training**. We retain `PC1` and `PC2` as a simple reduced representation. We also report how many components would be required to explain 95% of training variance, but do not assume two components are automatically optimal for forecasting.

In [ ]:
result = week5.fit_feature_reduction(splits, split_summary)
display(result.dimension_summary)
display(result.component_summary)
display(result.variance_summary)
display(result.loadings)
display(result.split_quality)
assert (result.split_quality["InvalidPCAValues"] == 0).all()
assert (result.split_quality["RowsBefore"] == result.split_quality["RowsAfter"]).all()

## 6. Synthetic augmentation decision

**No synthetic observations were added.** 
- Existing BTS dataset is already large, and artificial passenger demand could distort known seasonal dynamics, route discontinuities, and the 2020 demand-regime shift. 
- Synthetic tabular augmentation would require separate realism and utility testing and be generated only inside training. 
- Task explicitly allows creating cross-field features **and/or** using augmentation, while requiring dimensionality reduction.

In [ ]:
print("Synthetic data added: 0 rows")
print("PCA fitted on training rows only:", len(splits["Training"]))
print(f"Two PCA components retain {result.retained_variance_percent:.2f}% of standardized training variance.")
print("Number of PCs required for 95% training variance:", result.components_for_95pct)

## 7. Save compact audit tables, figures, samples

- Full 1.5-million-row outputs are **not** pushed to GitHub by default. 
- Small samples and precise fit metadata provide transparent evidence and allow results to be regenerated. 
- PCA outputs alone may lose important predictive signals.

In [ ]:
week5.save_outputs(result, PROJECT_ROOT)
print("Tables:", TABLES_DIR)
print("Figures:", FIGURES_DIR)
print("Samples:", PROCESSED_DIR)
for path in sorted(TABLES_DIR.glob("*")):
    print("  ", path.name)
display(result.reduced_samples["Training"].head())

## 8. Inspect training-only PCA charts

In [ ]:
for name in [
    "week5_figure_pca_explained_variance.png",
    "week5_figure_pca_loadings.png",
    "week5_figure_dimension_comparison.png",
]:
    path = FIGURES_DIR / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))

## 9. Final validation and completion

- This is feature engineering and dimensionality reduction
- This is **not a fitted passenger-demand forecasting model**. 
- The test labels were never used to select inputs or components. 
- The fraction of *training variance* explained by PCA is not a forecast accuracy metric.

In [ ]:
assert (TABLES_DIR / "week5_transform_metadata.json").is_file()
assert (FIGURES_DIR / "week5_figure_pca_explained_variance.png").is_file()
for name in ("training", "validation", "test"):
    assert (PROCESSED_DIR / f"week5_{name}_reduced_sample.csv").is_file()
assert len(result.kept_features) > 2
print("All Week 5 checks passed.")
print("===== WEEK 5 END =====")